# 1. Data Loading & Exploration

**Purpose:** Load raw spatial data, inspect its structure, and verify it's ready for segmentation.  
**Container:** Must be run inside `seg_sin_V1.sif` (JupyterLab or interactive session).  

This notebook is for **exploration only** — actual segmentation runs should be submitted as SLURM jobs.

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import sopa
import spatialdata
from spatialdata_io import xenium
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=100, frameon=False)
%matplotlib inline

print(f"sopa: {sopa.__version__}")
print(f"spatialdata: {spatialdata.__version__}")
print(f"scanpy: {sc.__version__}")

## Load pipeline config
All paths and parameters live in one YAML file.

In [ ]:
CONFIG_PATH = "../config/pipeline_config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

SAMPLE_ID = cfg["sample"]["id"]
PLATFORM = cfg["sample"]["platform"]
RAW_DATA = cfg["sample"]["raw_data_path"]
BASE_OUT = cfg["paths"]["base_output_dir"]

print(f"Sample:   {SAMPLE_ID}")
print(f"Platform: {PLATFORM}")
print(f"Raw data: {RAW_DATA}")

## Load raw spatial data

In [ ]:
sdata = xenium(RAW_DATA, cells_as_circles=True)
sdata

In [ ]:
# What's in the spatial data object?
print("Images:", list(sdata.images.keys()))
print("Labels:", list(sdata.labels.keys()) if hasattr(sdata, 'labels') else "none")
print("Points:", list(sdata.points.keys()) if hasattr(sdata, 'points') else "none")
print("Shapes:", list(sdata.shapes.keys()) if hasattr(sdata, 'shapes') else "none")

## Inspect morphology image

In [ ]:
# Plot the morphology image (DAPI)
if "morphology_focus" in sdata.images:
    img = sdata.images["morphology_focus"]
    print(f"Image shape: {img.shape}")
    print(f"Image dtype: {img.dtype}")
    
    # Quick thumbnail
    sdata.pl.render_images("morphology_focus").pl.show("global", figsize=(10, 10))
else:
    print("Available images:", list(sdata.images.keys()))

## Inspect transcripts

In [ ]:
# Transcript table overview
for key in sdata.points:
    pts = sdata.points[key]
    df = pts.compute() if hasattr(pts, 'compute') else pts
    print(f"\n{key}: {len(df):,} transcripts")
    print(f"  Columns: {list(df.columns)}")
    if 'feature_name' in df.columns:
        n_genes = df['feature_name'].nunique()
        print(f"  Unique genes: {n_genes}")
        print(f"  Top 10 genes:")
        print(df['feature_name'].value_counts().head(10).to_string())

## Inspect existing cell boundaries (if any)

In [ ]:
if sdata.shapes:
    for key in sdata.shapes:
        shapes = sdata.shapes[key]
        print(f"{key}: {len(shapes)} shapes")
        if hasattr(shapes, 'head'):
            display(shapes.head())
else:
    print("No shapes found (will be created by segmentation)")

## Tissue segmentation preview
SOPA's tissue detection identifies the tissue region to avoid processing empty areas.

In [ ]:
sopa.segmentation.tissue(sdata)

# Visualize tissue mask if available
if "tissue" in sdata.shapes:
    print(f"Tissue regions detected: {len(sdata.shapes['tissue'])}")
    sdata.pl.render_images("morphology_focus").pl.render_shapes(
        "tissue", outline_alpha=1, fill_alpha=0.1, outline_color="red"
    ).pl.show("global", figsize=(10, 10))

## Patching preview
Check how the image will be divided into patches for parallel processing.

In [ ]:
# Image patches
sopa.make_image_patches(sdata, patch_width=1200, patch_overlap=50)
print(f"Image patches: {len(sdata.shapes.get('image_patches', []))}")

# Transcript patches
sopa.make_transcript_patches(sdata, patch_width=500, prior_shapes_key="cell_boundaries")
print(f"Transcript patches: {len(sdata.shapes.get('transcript_patches', []))}")

## Save as .zarr for fast re-loading
Optional: Save a preprocessed zarr store so segmentation jobs load faster.

In [ ]:
# Uncomment to save:
# zarr_path = f"{BASE_OUT}/{SAMPLE_ID}/preprocessed.zarr"
# os.makedirs(os.path.dirname(zarr_path), exist_ok=True)
# sdata.write(zarr_path)
# print(f"Saved: {zarr_path}")

---
## Next steps

Data looks good? Submit segmentation jobs:

```bash
# From the project root:
python launch_pipeline.py --config config/pipeline_config.yaml --submit
```

Then explore results in **notebook 02 (per-method)** and **notebook 03 (QC/comparison)**.